# 01 — Stage-1 Multi-Label Pathology Triage & Glaucoma ONH Training (Google Colab TPU v2-8 / v5e-1)
Optimized for **Google Colab TPU (`torch_xla`)** using native `bfloat16` mixed precision, static tensor shapes (`drop_last=True` to avoid XLA graph recompilation), and `ColabTPUAccelerator`.
Trains `StrongCNN`, `ConvNeXtV2CoordinateTriage` (independent Sigmoid heads with `AsymmetricFocalLoss`), and `OpticNerveHeadUNet` (vCDR & ISNT segmentation).

In [ ]:
# 1. Mount Google Drive & Initialize Colab TPU Accelerator (torch_xla)
import os, sys
import torch

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')

from src.utils.tpu_accelerator import ColabTPUAccelerator

accel = ColabTPUAccelerator(prefer_tpu=True)
print(f'Active Accelerator: {accel.info.device_name} | Backend: {accel.info.backend} | Precision: {accel.info.amp_dtype}')

In [ ]:
# 2. Initialize Stage-1 Multi-Label Ensemble & Asymmetric Focal Loss on TPU
from src.tier2_triage import MultiLabelTriageEnsemble
from src.losses import AsymmetricFocalLoss

model = MultiLabelTriageEnsemble(base_channels=32).to(accel.device)
criterion = AsymmetricFocalLoss(gamma_neg=4.0, gamma_pos=1.0, clip=0.05).to(accel.device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

# Demo static-shape batch [DR, Glaucoma, Cataract] (drop_last=True prevents XLA recompilation)
x_batch = torch.randn(8, 3, 512, 512, device=accel.device)
y_multi = torch.tensor([
    [1.0, 0.0, 1.0], [0.0, 1.0, 0.0], [0.0, 0.0, 0.0], [1.0, 1.0, 0.0],
    [1.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, 1.0, 1.0], [0.0, 0.0, 0.0]
], device=accel.device)

optimizer.zero_grad()
with accel.autocast_context(enabled=True):
    outputs = model(x_batch)
    loss = 0.5 * criterion(outputs['logits_strong'], y_multi) + 0.5 * criterion(outputs['logits_convnext'], y_multi)

loss.backward()
accel.optimizer_step(optimizer)
print(f'Stage-1 Multi-Label ASL Loss (TPU bfloat16): {loss.item():.4f}')
accel.save_checkpoint(model, 'checkpoints/stage1_multi_label_best.pth')